In [ ]:
#pip install pandas numpy matplotlib seaborn scikit-learn torch jupyter pyarrow fastparquet

# Load Dataset

In [ ]:
import pandas as pd
import numpy as np

# Large-ST — adjust path/format to your file
traffic = pd.read_csv('large_st.csv')  
# Expected columns: timestamp, sensor_id, speed, flow, occupancy

# LA Metro monthly ridership — from NTD or Metro open data
ridership = pd.read_csv('la_metro_ridership.csv')
# Expected columns: year, month, route, boardings

# Align temporal granularity

In [ ]:
traffic['timestamp'] = pd.to_datetime(traffic['timestamp'])
traffic['year']  = traffic['timestamp'].dt.year
traffic['month'] = traffic['timestamp'].dt.month

# Build a monthly congestion index per region
# Congestion = average % of time speed drops below a threshold
SPEED_THRESHOLD = 45  # mph — below this = congested

traffic['is_congested'] = (traffic['speed'] < SPEED_THRESHOLD).astype(int)

monthly_congestion = traffic.groupby(['year', 'month']).agg(
    avg_speed        = ('speed', 'mean'),
    avg_flow         = ('flow', 'mean'),
    congestion_pct   = ('is_congested', 'mean'),  # 0–1, proportion of hours congested
    peak_congestion  = ('is_congested', 'max')
).reset_index()

# Merge with ridership on year + month
df = pd.merge(monthly_congestion, ridership, on=['year', 'month'], how='inner')
print(df.shape)
print(df.head())

# Feature Engineering

In [ ]:
# Congestion severity tiers — useful for groupby analysis
df['congestion_tier'] = pd.cut(
    df['congestion_pct'],
    bins=[0, 0.2, 0.4, 0.6, 1.0],
    labels=['low', 'moderate', 'high', 'severe']
)

# Lag features — does last month's congestion predict this month's ridership?
df = df.sort_values(['year', 'month'])
df['congestion_lag1'] = df['congestion_pct'].shift(1)
df['ridership_lag1']  = df['boardings'].shift(1)

# Season — useful control variable
df['season'] = df['month'].map({
    12:'winter', 1:'winter', 2:'winter',
    3:'spring',  4:'spring', 5:'spring',
    6:'summer',  7:'summer', 8:'summer',
    9:'fall',   10:'fall',  11:'fall'
})

df = df.dropna()

# EDA & Key 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot 1: Ridership vs. congestion scatter — your hero chart
plt.figure(figsize=(8, 5))
plt.scatter(df['congestion_pct'], df['boardings'], alpha=0.6)
plt.xlabel('Monthly congestion rate')
plt.ylabel('Total boardings')
plt.title('Do people take transit when roads are congested?')
# Add a trend line
z = np.polyfit(df['congestion_pct'], df['boardings'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['congestion_pct'].min(), df['congestion_pct'].max(), 100)
plt.plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Trend')
plt.legend()
plt.savefig('congestion_vs_ridership.png', dpi=150, bbox_inches='tight')

# Plot 2: Average ridership by congestion tier
tier_avg = df.groupby('congestion_tier')['boardings'].mean()
tier_avg.plot(kind='bar', color=['#5DCAA5','#EF9F27','#D85A30','#E24B4A'])
plt.title('Average ridership by congestion severity')
plt.ylabel('Average monthly boardings')
plt.xticks(rotation=0)
plt.savefig('ridership_by_tier.png', dpi=150, bbox_inches='tight')

# Plot 3: Time series — both signals on one chart
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.plot(df.index, df['congestion_pct'], color='#D85A30', label='Congestion rate')
ax2.plot(df.index, df['boardings'],      color='#1D9E75', label='Ridership', linestyle='--')
ax1.set_ylabel('Congestion rate', color='#D85A30')
ax2.set_ylabel('Monthly boardings', color='#1D9E75')
plt.title('Congestion rate vs. transit ridership over time')
plt.savefig('dual_timeseries.png', dpi=150, bbox_inches='tight')

# Modeling

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode season
le = LabelEncoder()
df['season_enc'] = le.fit_transform(df['season'])

features = ['congestion_pct', 'avg_speed', 'avg_flow', 
            'congestion_lag1', 'ridership_lag1', 'season_enc']
target   = 'boardings'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # time-based split
)

# Baseline
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Main model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Results table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R²':    [r2_score(y_test, lr_preds),  r2_score(y_test, rf_preds)],
    'MAE':   [mean_absolute_error(y_test, lr_preds), mean_absolute_error(y_test, rf_preds)]
})
print(results)

# Feature importance — great slide visual
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
importances.plot(kind='barh')
plt.title('What drives transit ridership changes?')
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')

In [ ]:
from scipy import stats

r, p = stats.pearsonr(df['congestion_pct'], df['boardings'])
print(f"Correlation: r = {r:.3f}, p = {p:.4f}")
# e.g. "r = 0.62, p < 0.001 — congestion explains 38% of ridership variance"